# Crop augmented spot archives from 384 to 128

This notebook reads already generated HDF5 spot-separation archives and writes new archives where every sample is center-cropped from `384 x 384` to `128 x 128`.

No resizing or interpolation is used. The notebook simply cuts equal margins from the edges and keeps the central `128 x 128` pixels.

In [ ]:
from pathlib import Path
import json

import h5py
import numpy as np


def find_project_dir():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "train.py").exists() and (candidate / "augment_data").exists():
            return candidate
    raise RuntimeError("Could not find the project directory containing train.py.")


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR.parent / "data_esrf"

CROP_SIZE = 128
EXPECTED_INPUT_SHAPE = (384, 384)
CROP_DATASETS = {"image", "spot_images", "spot_masks"}

# These are the names used by augmentation.ipynb. The older/default training file is
# included as a fallback because it exists in this workspace.
INPUT_FILES = [
    DATA_DIR / "augmented_spots_train.h5",
    DATA_DIR / "augmented_spots_validation.h5",
    DATA_DIR / "augmented_spot_patches.h5",
]

OUTPUT_SUFFIX = "_crop128"
OVERWRITE = False
RUN_CROPPING = False

existing_inputs = [path for path in INPUT_FILES if path.exists()]
if not existing_inputs:
    raise FileNotFoundError(f"None of the configured input files exist in {DATA_DIR}")

print(f"Project: {PROJECT_DIR}")
print("Input files:")
for path in existing_inputs:
    print("  ", path)

## Cropping helpers

In [ ]:
def output_path_for(input_path, suffix=OUTPUT_SUFFIX):
    input_path = Path(input_path)
    return input_path.with_name(f"{input_path.stem}{suffix}{input_path.suffix}")


def sample_group_names(h5_file):
    return sorted(
        name for name, obj in h5_file.items()
        if isinstance(obj, h5py.Group)
        and "image" in obj
        and "spot_images" in obj
    )


def center_crop(array, crop_size=CROP_SIZE, expected_shape=EXPECTED_INPUT_SHAPE):
    array = np.asarray(array)
    if array.ndim < 2:
        raise ValueError(f"Cannot crop an array with shape {array.shape}")

    height, width = array.shape[-2:]
    if expected_shape is not None and (height, width) != tuple(expected_shape):
        raise ValueError(
            f"Expected trailing image shape {expected_shape}, got {(height, width)} "
            f"for array shape {array.shape}"
        )
    if height < crop_size or width < crop_size:
        raise ValueError(f"Crop size {crop_size} is larger than array shape {array.shape}")

    row0 = (height - crop_size) // 2
    col0 = (width - crop_size) // 2
    return array[..., row0:row0 + crop_size, col0:col0 + crop_size]


def create_dataset_like(output_group, name, source_dataset, data):
    kwargs = {}
    if data.ndim >= 2:
        kwargs["compression"] = "gzip"
    dataset = output_group.create_dataset(name, data=data, dtype=source_dataset.dtype, **kwargs)
    for key, value in source_dataset.attrs.items():
        dataset.attrs[key] = value
    return dataset


def copy_attrs(source, target):
    for key, value in source.attrs.items():
        target.attrs[key] = value

## Inspect configured inputs

In [ ]:
for input_path in existing_inputs:
    with h5py.File(input_path, "r") as f:
        names = sample_group_names(f)
        if not names:
            print(f"{input_path.name}: no sample groups found")
            continue
        first = f[names[0]]
        print(f"{input_path.name}: {len(names):,} samples")
        for dataset_name in sorted(CROP_DATASETS & set(first.keys())):
            print(f"  {dataset_name}: {first[dataset_name].shape} -> ", end="")
            print(center_crop(first[dataset_name][()]).shape)
        print(f"  output: {output_path_for(input_path)}")

## Write cropped archives

Set `RUN_CROPPING = True` in the first code cell when the inspection output looks right.

In [ ]:
def crop_archive(input_path, output_path=None, overwrite=OVERWRITE):
    input_path = Path(input_path)
    output_path = output_path_for(input_path) if output_path is None else Path(output_path)
    temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")

    if output_path.exists() and not overwrite:
        raise FileExistsError(f"Output exists already: {output_path}")
    if temporary_path.exists():
        temporary_path.unlink()

    with h5py.File(input_path, "r") as source, h5py.File(temporary_path, "w") as target:
        copy_attrs(source, target)
        target.attrs["crop_size"] = int(CROP_SIZE)
        target.attrs["crop_policy"] = "center crop only; no resizing or interpolation"
        target.attrs["source_archive"] = str(input_path)
        target.attrs["original_patch_shape"] = json.dumps(list(EXPECTED_INPUT_SHAPE))
        target.attrs["cropped_patch_shape"] = json.dumps([CROP_SIZE, CROP_SIZE])

        samples = set(sample_group_names(source))
        if not samples:
            raise ValueError(f"No sample groups found in {input_path}")

        for name, item in source.items():
            if name not in samples:
                source.copy(name, target)
                continue

            source_group = item
            target_group = target.create_group(name)
            copy_attrs(source_group, target_group)

            for dataset_name, dataset in source_group.items():
                if dataset_name in CROP_DATASETS:
                    data = center_crop(dataset[()])
                    create_dataset_like(target_group, dataset_name, dataset, data)
                else:
                    source_group.copy(dataset_name, target_group)

    temporary_path.replace(output_path)
    return output_path


if RUN_CROPPING:
    for input_path in existing_inputs:
        output_path = crop_archive(input_path)
        print(f"Wrote {output_path}")
else:
    print("Cropping not started. Set RUN_CROPPING = True to write the cropped archives.")

## Validate cropped files

In [ ]:
def validate_cropped_archive(path, checks=10):
    path = Path(path)
    if not path.exists():
        print(f"Not written yet: {path}")
        return False

    with h5py.File(path, "r") as f:
        names = sample_group_names(f)
        if not names:
            raise ValueError(f"No sample groups found in {path}")
        indices = np.linspace(0, len(names) - 1, min(checks, len(names)), dtype=int)
        for index in indices:
            group = f[names[index]]
            assert group["image"].shape == (CROP_SIZE, CROP_SIZE), group["image"].shape
            assert group["spot_images"].shape[-2:] == (CROP_SIZE, CROP_SIZE), group["spot_images"].shape
            if "spot_masks" in group:
                assert group["spot_masks"].shape[-2:] == (CROP_SIZE, CROP_SIZE), group["spot_masks"].shape
    print(f"Valid: {path} ({len(names):,} samples)")
    return True


for input_path in existing_inputs:
    validate_cropped_archive(output_path_for(input_path))